# Animate a cartoon character picture (free, on Kaggle's GPU)

Turns one still picture of a character into a short moving clip with **LTX-Video** (open source, Apache-2.0 code).
Needs: Settings -> Accelerator -> **GPU T4 x2**, and Settings -> **Internet on** (phone-verified Kaggle account).
1. Run the cells top to bottom. 2. Change `PICTURE_URL` and `PROMPT` for a new shot. 3. Download the mp4 from Output.

In [ ]:
!pip -q install -U diffusers transformers accelerate sentencepiece imageio imageio-ffmpeg

In [ ]:
import torch, requests, io
from PIL import Image
from diffusers import LTXImageToVideoPipeline
from diffusers.utils import export_to_video

# The picture to animate (must be a direct link to a PNG/JPG; 512x768 portrait works best)
PICTURE_URL = "https://raw.githubusercontent.com/ayanrindeisraeloluwabusayo-prog/website/claude/taste-skill-repo-2olaxk/videos/cartoon-cast/kaggle/regular_shrug.png"
PROMPT = ("A 3D animated cartoon man, bald, with a long neck, big nose, white t-shirt and dark cargo trousers, stands in a grey studio. "
          "He shrugs his shoulders, tilts his head, raises his eyebrows, then lowers his arms and smiles. "
          "Smooth Pixar-style character animation, the camera does not move, soft studio lighting.")
NEGATIVE = "worst quality, blurry, jittery, distorted, deformed face, extra limbs, text, watermark, logo"

image = Image.open(io.BytesIO(requests.get(PICTURE_URL, timeout=60).content)).convert("RGB").resize((512, 768))
image

In [ ]:
pipe = LTXImageToVideoPipeline.from_pretrained("Lightricks/LTX-Video", torch_dtype=torch.bfloat16)
pipe.enable_model_cpu_offload()      # keeps the big parts off the GPU until they are needed (T4 has 16 GB)
pipe.vae.enable_tiling()

In [ ]:
generator = torch.Generator("cpu").manual_seed(7)
video = pipe(image=image, prompt=PROMPT, negative_prompt=NEGATIVE, width=512, height=768,
             num_frames=97, num_inference_steps=40, guidance_scale=3.0, generator=generator).frames[0]
export_to_video(video, "/kaggle/working/clip.mp4", fps=24)
print("saved /kaggle/working/clip.mp4  (about 4 seconds)")

Tips: one action per clip (4 s); describe the motion plainly; keep "the camera does not move" for clean edits.
If it runs out of memory, lower `num_frames` to 65 or the size to 448x672.